# Teste Linux do protótipo no Google Colab
Envie o ZIP da raiz do projeto. O notebook executa o mesmo `run.sh` entregue ao avaliador.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded))
assert zip_name.lower().endswith('.zip'), 'Envie o projeto em um arquivo ZIP.'

In [ ]:
import pathlib, shutil
uploaded_zip = pathlib.Path(zip_name).resolve()
workspace = pathlib.Path('/content/prototipo_tickets')
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir()
shutil.unpack_archive(uploaded_zip, workspace)
matches = list(workspace.rglob('submissions/ramon-baptista/solution/run.sh'))
assert len(matches) == 1, f'Esperava um run.sh; encontrei {len(matches)}.'
PROJECT = matches[0].parent
print('Projeto:', PROJECT)

## Teste automatizado do launcher Linux
Cria um ambiente isolado, instala as dependências (incluindo Flask), reproduz a avaliação pelo terminal (`--cli`) e roda a suíte completa de testes, inclusive os da interface web.

In [ ]:
import subprocess, time
start = time.perf_counter()
test = subprocess.run(['sh', str(PROJECT / 'run.sh'), '--cli'], input='2\n0\n', text=True, capture_output=True, cwd=PROJECT)
print(test.stdout)
if test.stderr:
    print('AVISOS/ERROS (stderr):\n' + test.stderr)
print(f'Tempo total: {time.perf_counter() - start:.1f} s')
assert test.returncode == 0, f'run.sh terminou com código {test.returncode}'
assert 'A reprodução passou' in test.stdout

In [ ]:
import os
py = str(PROJECT / '.venv' / 'bin' / 'python')
suite = subprocess.run([py, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], text=True, capture_output=True, cwd=PROJECT, env={**os.environ, 'PYTHONPATH': str(PROJECT / 'src')})
print(suite.stderr[-3000:])
assert suite.returncode == 0, 'A suíte de testes falhou.'


## Menu interativo
Use os exemplos do README. Para sair, digite `0`.

In [ ]:
import os, sys
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)
from ticket_classifier.cli import main
main()